In [1]:
"""
Library availability checker for Qwen3.5-9B finetuning pipeline.
Tries to import every relevant library and prints a summary table.
"""

import importlib
import sys
from typing import Optional

# ── colour helpers (no external deps) ──────────────────────────────────────
GREEN  = "\033[92m"
RED    = "\033[91m"
YELLOW = "\033[93m"
CYAN   = "\033[96m"
BOLD   = "\033[1m"
RESET  = "\033[0m"

def ok(s):    return f"{GREEN}✔  {s}{RESET}"
def fail(s):  return f"{RED}✘  {s}{RESET}"
def info(s):  return f"{CYAN}{s}{RESET}"
def header(s):return f"{BOLD}{CYAN}{s}{RESET}"

# ── library manifest ────────────────────────────────────────────────────────
# Each entry: (import_name, pip_install_name, category, priority)
#   priority: "must" | "recommended" | "optional"
LIBRARIES = [
    # ── Core finetuning stack ──────────────────────────────────────────────
    ("torch",               "torch",                        "Core",        "must"),
    ("transformers",        "transformers>=4.51",           "Core",        "must"),
    ("peft",                "peft",                         "Core",        "must"),
    ("trl",                 "trl",                          "Core",        "must"),
    ("accelerate",          "accelerate",                   "Core",        "must"),
    ("bitsandbytes",        "bitsandbytes",                 "Core",        "optional"),
    ("unsloth",             "unsloth",                      "Core",        "recommended"),

    # ── Data & datasets ────────────────────────────────────────────────────
    ("datasets",            "datasets",                     "Data",        "must"),
    ("jsonschema",          "jsonschema",                   "Data",        "must"),

    # ── Training infrastructure ────────────────────────────────────────────
    ("deepspeed",           "deepspeed",                    "Infra",       "recommended"),

    # ── Experiment tracking ────────────────────────────────────────────────
    ("wandb",               "wandb",                        "Tracking",    "recommended"),

    # ── Inference & serving ────────────────────────────────────────────────
    ("vllm",                "vllm>=0.17.0",                 "Inference",   "recommended"),

    # ── Evaluation ────────────────────────────────────────────────────────
    ("evaluate",            "evaluate",                     "Evaluation",  "optional"),
    ("sacrebleu",           "sacrebleu",                    "Evaluation",  "optional"),

    # ── Teacher-model / synthetic data APIs ───────────────────────────────
    ("openai",              "openai",                       "APIs",        "must"),
    ("anthropic",           "anthropic",                    "APIs",        "must"),
]

# ── sub-module imports to verify (checked only if parent is present) ────────
SUB_IMPORTS = {
    "transformers": [
        "AutoModelForCausalLM", "AutoTokenizer", "BitsAndBytesConfig",
        "TrainingArguments", "GenerationConfig", "pipeline",
    ],
    "peft": [
        "LoraConfig", "get_peft_model", "PeftModel", "TaskType",
        "prepare_model_for_kbit_training", "AutoPeftModelForCausalLM",
    ],
    "trl": [
        "SFTTrainer", "SFTConfig", "DPOTrainer", "DPOConfig",
        "PPOTrainer", "GRPOTrainer", "DataCollatorForCompletionOnlyLM",
    ],
    "datasets": [
        "load_dataset", "Dataset", "DatasetDict",
        "concatenate_datasets", "interleave_datasets",
    ],
    "accelerate": [
        "Accelerator", "DistributedType",
    ],
    "bitsandbytes": [],   # checked at top-level only
    "torch": [],          # checked at top-level only
}

# ── version checks (warn if below minimum) ──────────────────────────────────
MIN_VERSIONS = {
    "transformers": (4, 51, 0),
    "vllm":         (0, 17, 0),
    "torch":        (2,  0, 0),
}

# ──────────────────────────────────────────────────────────────────────────
def parse_version(v: str) -> tuple:
    """Turn '4.51.0.dev0' → (4, 51, 0)."""
    parts = []
    for seg in v.split(".")[:3]:
        seg = "".join(c for c in seg if c.isdigit())
        parts.append(int(seg) if seg else 0)
    while len(parts) < 3:
        parts.append(0)
    return tuple(parts)

def try_import(module_name: str) -> tuple[bool, Optional[str], Optional[str]]:
    """
    Returns (success, version_string, error_message).
    """
    try:
        mod = importlib.import_module(module_name)
        version = getattr(mod, "__version__", None) or getattr(mod, "version", None)
        return True, (str(version) if version else "unknown"), None
    except ImportError as e:
        return False, None, str(e)
    except Exception as e:
        return False, None, f"unexpected error: {e}"

def check_sub_imports(parent: str, names: list[str]) -> dict[str, bool]:
    results = {}
    try:
        mod = importlib.import_module(parent)
        for name in names:
            results[name] = hasattr(mod, name)
    except Exception:
        for name in names:
            results[name] = False
    return results

def version_ok(module: str, version_str: str) -> Optional[str]:
    """Returns a warning string if version is below minimum, else None."""
    if module not in MIN_VERSIONS or version_str == "unknown":
        return None
    try:
        actual = parse_version(version_str)
        minimum = MIN_VERSIONS[module]
        if actual < minimum:
            min_str = ".".join(map(str, minimum))
            return f"version {version_str} < required {min_str}"
    except Exception:
        pass
    return None

# ── main check ───────────────────────────────────────────────────────────────
def run_checks():
    print()
    print(header("=" * 65))
    print(header("  Qwen3.5-9B Finetuning — Library Availability Check"))
    print(header("=" * 65))
    print(f"  Python  : {sys.version.split()[0]}  |  {sys.executable}")
    print()

    results = []   # (name, pip_name, category, priority, installed, version, warning, missing_subs)

    for import_name, pip_name, category, priority in LIBRARIES:
        installed, version, err = try_import(import_name)

        warning = None
        if installed:
            warning = version_ok(import_name, version)

        missing_subs = []
        if installed and import_name in SUB_IMPORTS and SUB_IMPORTS[import_name]:
            sub_results = check_sub_imports(import_name, SUB_IMPORTS[import_name])
            missing_subs = [k for k, v in sub_results.items() if not v]

        results.append((import_name, pip_name, category, priority,
                        installed, version, warning, missing_subs))

    # ── per-library output ─────────────────────────────────────────────────
    current_category = None
    for (import_name, pip_name, category, priority,
         installed, version, warning, missing_subs) in results:

        if category != current_category:
            current_category = category
            print(f"  {BOLD}── {category} {'─' * (50 - len(category))}{RESET}")

        ver_str = f"v{version}" if version and version != "unknown" else ""
        priority_tag = {"must": f"{RED}[must]{RESET}",
                        "recommended": f"{YELLOW}[rec]{RESET}",
                        "optional": f"{CYAN}[opt]{RESET}"}[priority]

        if installed:
            status = ok(f"{import_name:<20} {ver_str:<14} {priority_tag}")
        else:
            status = fail(f"{import_name:<20} {'NOT FOUND':<14} {priority_tag}")

        print(f"  {status}")

        if installed and warning:
            print(f"    {YELLOW}⚠  {warning}{RESET}")

        if installed and missing_subs:
            print(f"    {YELLOW}⚠  missing sub-symbols: {', '.join(missing_subs)}{RESET}")

        if not installed:
            print(f"    {YELLOW}→  pip install {pip_name}{RESET}")

    # ── summary table ──────────────────────────────────────────────────────
    installed_libs  = [(n, pip, cat, pri, ver) for n, pip, cat, pri, ok_, ver, *_ in results if ok_]
    missing_libs    = [(n, pip, cat, pri)      for n, pip, cat, pri, ok_, *_ in results if not ok_]
    warned_libs     = [(n, w)                  for n, pip, cat, pri, ok_, ver, w, ms in results if ok_ and (w or ms)]

    col = [64, 30, 14, 14, 10]   # column widths

    def row(*cells):
        return "  " + "  ".join(str(c).ljust(w) for c, w in zip(cells, col))

    divider = "  " + "-" * (sum(col) + 2 * len(col))

    print()
    print(header("=" * 65))
    print(header("  Summary"))
    print(header("=" * 65))

    # installed
    if installed_libs:
        print()
        print(f"  {BOLD}{GREEN}Installed ({len(installed_libs)}){RESET}")
        print(divider)
        print(row("Library", "pip package", "Category", "Priority", "Version"))
        print(divider)
        for n, pip, cat, pri, ver in installed_libs:
            print(row(n, pip, cat, pri, ver or "?"))
        print(divider)

    # needs install
    if missing_libs:
        print()
        print(f"  {BOLD}{RED}Needs installation ({len(missing_libs)}){RESET}")
        print(divider)
        print(row("Library", "Install command", "Category", "Priority", ""))
        print(divider)
        for n, pip, cat, pri in missing_libs:
            print(row(n, f"pip install {pip}", cat, pri, ""))
        print(divider)

        # one-liner to install all missing
        must_missing = [pip for n, pip, cat, pri in missing_libs if pri == "must"]
        rec_missing  = [pip for n, pip, cat, pri in missing_libs if pri == "recommended"]

        print()
        if must_missing:
            cmd = "pip install " + " ".join(must_missing)
            print(f"  {BOLD}{RED}Install all must-haves:{RESET}")
            print(f"  {YELLOW}{cmd}{RESET}")
        if rec_missing:
            cmd = "pip install " + " ".join(rec_missing)
            print(f"  {BOLD}{YELLOW}Install all recommended:{RESET}")
            print(f"  {YELLOW}{cmd}{RESET}")

    # warnings
    if warned_libs:
        print()
        print(f"  {BOLD}{YELLOW}Warnings ({len(warned_libs)}){RESET}")
        for n, w in warned_libs:
            print(f"  {YELLOW}⚠  {n}: {w}{RESET}")

    print()
    total    = len(results)
    n_ok     = len(installed_libs)
    n_miss   = len(missing_libs)
    n_warn   = len(warned_libs)
    print(f"  {BOLD}Total: {total}  |  {GREEN}Installed: {n_ok}{RESET}  |  "
          f"{RED}Missing: {n_miss}{RESET}  |  {YELLOW}Warnings: {n_warn}{RESET}")
    print()

    return {
        "installed": installed_libs,
        "missing":   missing_libs,
        "warnings":  warned_libs,
    }


# ── entry point ───────────────────────────────────────────────────────────────
if __name__ == "__main__":
    run_checks()


  Qwen3.5-9B Finetuning — Library Availability Check
  Python  : 3.12.12  |  /Users/ankitwahane/Developer/TFM-Benchmarking/.venv/bin/python

  ── Core ──────────────────────────────────────────────
  ✘  torch                NOT FOUND      [must]
    →  pip install torch
  ✘  transformers         NOT FOUND      [must]
    →  pip install transformers>=4.51
  ✘  peft                 NOT FOUND      [must]
    →  pip install peft
  ✘  trl                  NOT FOUND      [must]
    →  pip install trl
  ✘  accelerate           NOT FOUND      [must]
    →  pip install accelerate
  ✘  bitsandbytes         NOT FOUND      [opt]
    →  pip install bitsandbytes
  ✘  unsloth              NOT FOUND      [rec]
    →  pip install unsloth
  ── Data ──────────────────────────────────────────────
  ✘  datasets             NOT FOUND      [must]
    →  pip install datasets
  ✘  jsonschema           NOT FOUND      [must]
    →  pip install jsonschema
  ── Infra ─────────────────────────────────────────────
